<a href="https://colab.research.google.com/github/Sayak-coder/SIH_26086/blob/main/weekly_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install tensorflow

In [2]:
!pip install --upgrade xee

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 1.8 MB/s eta 0:00:00


In [3]:
!pip install -u geemap


Usage:   
  pip3 install [options] <requirement specifier> [package-index-options] ...
  pip3 install [options] -r <requirements file> [package-index-options] ...
  pip3 install [options] [-e] <vcs project url> ...
  pip3 install [options] [-e] <local project path> ...
  pip3 install [options] <archive url/path> ...

no such option: -u


In [4]:
!pip install xee

In [24]:
import ee
import geemap
import xarray as xr
from xee import helpers
import shapely.geometry
import torch
from sklearn.preprocessing import StandardScaler
import tensorflow

In [6]:
ee.Authenticate()
ee.Initialize(
    project='gen-lang-client-0960220624',
    opt_url='https://earthengine-highvolume.googleapis.com'
)

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


In [7]:
map=geemap.Map()
map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [9]:
roi=map.draw_last_feature.geometry()
roi

ee.Geometry({
  "functionInvocationValue": {
    "functionName": "Feature.geometry",
    "arguments": {
      "feature": {
        "functionInvocationValue": {
          "functionName": "Feature",
          "arguments": {
            "geometry": {
              "functionInvocationValue": {
                "functionName": "GeometryConstructors.Polygon",
                "arguments": {
                  "coordinates": {
                    "constantValue": [
                      [
                        [
                          -273.47168,
                          22.014361
                        ],
                        [
                          -273.47168,
                          25.740529
                        ],
                        [
                          -271.373291,
                          25.740529
                        ],
                        [
                          -271.373291,
                          22.014361
                        ],
                        [
                          -273.47168,
                          22.014361
                        ]
                      ]
                    ]
                  },
                  "geodesic": {
                    "constantValue": false
                  }
                }
              }
            }
          }
        }
      }
    }
  }
})

In [10]:
def wrap_lon(lon):
    return ((lon + 180) % 360) - 180

coords = roi.getInfo()['coordinates']
fixed_coords = [[[wrap_lon(lon), lat] for lon, lat in ring] for ring in coords]
roi_fixed = ee.Geometry.Polygon(fixed_coords)
roi_fixed

ee.Geometry({
  "functionInvocationValue": {
    "functionName": "GeometryConstructors.Polygon",
    "arguments": {
      "coordinates": {
        "constantValue": [
          [
            [
              86.52832000000001,
              22.014361
            ],
            [
              86.52832000000001,
              25.740529
            ],
            [
              88.626709,
              25.740529
            ],
            [
              88.626709,
              22.014361
            ],
            [
              86.52832000000001,
              22.014361
            ]
          ]
        ]
      },
      "evenOdd": {
        "constantValue": true
      }
    }
  }
})

In [11]:
start_time=ee.Date('2020')
end_time=ee.Date('2026')
time_diff=end_time.difference(start_time,'week')
time_diff

In [12]:
end_date = start_time.advance(1, 'week')
end_date

In [13]:
time_list = ee.List.sequence(0, ee.Number(time_diff).subtract(1)).map(
    lambda x: start_time.advance(x, 'week')
)
time_list

In [14]:
time_list = ee.List.sequence(0, ee.Number(time_diff).subtract(1)).map(
    lambda x: start_time.advance(x, 'week')
)

def daily(date, col, bands):
    d = ee.Date(date)
    blank = ee.Image.constant(ee.List.repeat(0, len(bands))).rename(bands).selfMask()
    img = col.filterDate(d, d.advance(1, 'week')).merge(ee.ImageCollection([blank])).mean()
    return img.set('system:time_start', d.millis())

pr = (
    ee.ImageCollection("UCSB-CHC/CHIRPS/V3/DAILY_RNL")
    .filterDate(start_time, end_time)
    .select(['precipitation'],['pr'])
)

veg = (
    ee.ImageCollection("MODIS/061/MOD13Q1")
    .filterDate(start_time,end_time)
    .select(['NDVI','EVI'],['ndvi','evi'])
)

temp=(
    ee.ImageCollection("MODIS/061/MOD11A1")
    .filterDate(start_time,end_time)
    .select(['LST_Day_1km','LST_Night_1km'],['dayT','nightT'])
)

et=(
    ee.ImageCollection("MODIS/061/MOD16A2")
    .filterDate(start_time,end_time)
    .select(['ET'],['et'])
)

pr_daily=ee.ImageCollection(
    time_list.map(
        lambda x:daily(x, pr,['pr'])
    )
)

veg_daily=ee.ImageCollection(
    time_list.map(
        lambda x:daily(x, veg,['ndvi','evi'])
    )
)

temp_daily=ee.ImageCollection(
    time_list.map(
        lambda x:daily(x, temp,['dayT','nightT'])
    )
)

et_daily=ee.ImageCollection(
    time_list.map(
        lambda x:daily(x, et,['et'])
    )
)


In [15]:
landcover=(
    ee.ImageCollection("MODIS/061/MCD12Q1").filterDate(start_time,end_time).mode().select(['LC_Type1'],['lc'])
)

topography=(
    ee.Image("USGS/GTOPO30").rename('dem')
)

In [16]:
landcover

In [17]:
# collection=temp_daily.combine(veg_daily).combine(et_daily).combine(pr_daily).map(
collection=temp_daily.combine(et_daily).combine(pr_daily).map(
    lambda x: x.addBands(landcover).addBands(topography)
)
collection

In [18]:
# roi_fixed is an ee.Geometry — convert it to GeoJSON, then to shapely
geojson_roi_fixed = roi_fixed.getInfo()
shapely_roi = shapely.geometry.shape(geojson_roi_fixed)

grid = helpers.fit_geometry(
    geometry=shapely_roi,
    grid_crs='EPSG:4326',
    grid_scale=(0.01, -0.01)
)
ds = xr.open_dataset(collection, engine='ee', **grid)

In [19]:
ds1km=ds

In [20]:
ds1km=ds1km.sortby('time')*1

In [21]:
ds1km

<xarray.Dataset> Size: 593MB
Dimensions:  (time: 313, y: 374, x: 211)
Coordinates:
  * time     (time) datetime64[ns] 3kB 2020-01-01 2020-01-08 ... 2025-12-24
  * y        (y) float64 3kB 25.75 25.73 25.73 25.71 ... 22.05 22.04 22.03 22.02
  * x        (x) float64 2kB 86.52 86.53 86.54 86.55 ... 88.59 88.6 88.61 88.62
Data variables:
    dayT     (time, y, x) float32 99MB 1.46e+04 1.459e+04 ... 1.464e+04 nan
    nightT   (time, y, x) float32 99MB 1.425e+04 1.428e+04 ... 1.46e+04
    et       (time, y, x) float32 99MB nan nan nan nan nan ... 43.0 38.0 nan nan
    pr       (time, y, x) float32 99MB 3.099 3.099 3.099 ... 0.02575 0.02575
    lc       (time, y, x) float32 99MB 12.0 12.0 12.0 12.0 ... 12.0 11.0 17.0
    dem      (time, y, x) float32 99MB 37.0 36.0 35.0 36.0 ... 13.0 13.0 1.0 nan

In [22]:
ds1km=ds1km.to_dataframe().dropna()

In [23]:
ds1km

dayT        nightT    et        pr    lc  \
time       y      x                                                          
2020-12-30 25.745 86.525  14709.500000  14222.000000  73.0  0.140409  12.0   
                  86.535  14717.500000  14200.000000  90.0  0.140409  12.0   
                  86.545  14672.666992  14191.666992  84.0  0.140409  12.0   
                  86.555  14668.333008  14196.000000  79.0  0.138287  12.0   
                  86.565  14663.333008  14200.666992  83.0  0.138287  12.0   
...                                ...           ...   ...       ...   ...   
2025-12-24 22.015 88.035  14562.000000  14374.833008  41.0  0.000000  12.0   
                  88.045  14551.500000  14439.500000  38.0  0.000000  16.0   
                  88.115  14568.000000  14436.333008  38.0  0.003979  11.0   
                  88.125  14568.000000  14424.500000  43.0  0.003979  11.0   
                  88.225  14505.000000  14336.333008  42.0  0.026698  12.0   

                           dem  
time       y      x             
2020-12-30 25.745 86.525  37.0  
                  86.535  36.0  
                  86.545  35.0  
                  86.555  36.0  
                  86.565  34.0  
...                        ...  
2025-12-24 22.015 88.035   6.0  
                  88.045   8.0  
                  88.115   4.0  
                  88.125   7.0  
                  88.225   5.0  

[11588397 rows x 6 columns]

In [25]:
X1km=ds1km.drop('pr',axis=1)
y1km=ds1km['pr']

In [28]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X1km,y1km,test_size=0.2,random_state=45)

In [30]:
scaler=StandardScaler()

X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

y_train=scaler.fit_transform(y_train.values.reshape(-1,1))
y_test=scaler.transform(y_test.values.reshape(-1,1))

In [31]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [34]:
model=Sequential()
model.add(Dense(64,activation='relu',input_dim=X_train.shape[1]))
model.add(Dense(32,activation='relu'))
model.add(Dense(16,activation='relu'))
model.add(Dense(1,activation='relu'))
model.compile(
    loss='mse',
    optimizer='adam',
    metrics=['mae']
)

In [38]:
from re import VERBOSE
model.fit(
    X_train,y_train,epochs=2,batch_size=8,validation_split=0.2,verbose=1
)

Epoch 1/2
927072/927072 ━━━━━━━━━━━━━━━━━━━━ 2376s 3ms/step - loss: 1.0005 - mae: 0.6169 - val_loss: 0.9981 - val_mae: 0.6168
Epoch 2/2
927072/927072 ━━━━━━━━━━━━━━━━━━━━ 2357s 3ms/step - loss: 1.0005 - mae: 0.6169 - val_loss: 0.9981 - val_mae: 0.6168


In [39]:
model.evaluation=model.evaluate(X_test,y_test)

72428/72428 ━━━━━━━━━━━━━━━━━━━━ 170s 2ms/step - loss: 1.0029 - mae: 0.6169


In [41]:
ds1km['pr1km']=model.predict(X_test)

72428/72428 ━━━━━━━━━━━━━━━━━━━━ 102s 1ms/step


ValueError: Length of values (2317680) does not match length of index (11588397)

In [40]:
result_sub=result.sel(time='2024')

NameError: name 'result' is not defined